Controle de qualidade no pré-processamento de imagens médicas
* Verificação automática da qualidade e consistência das imagens de Tomografia Computadorizada (TC) e das respectivas máscaras tumorais
* Objetivo: Analisar os 421 pacientes da coorte NSCLC-Radiomics (Lung1), verificando aspectos como alinhamento espacial, dimensões das imagens e características dos valores dos exames.

Arquivos que foram gerados durante o pré-processamento de imagem:
* `ct_aligned.nii.gz` e `seg_aligned.nii.gz`: conversão de formato DICOM -> NIfTI (*Neuroimaging Informatics Technology Initiative*), garantindo o alinhamento espacial (mesma origem, direção e matriz) entre a tomografia e a segmentação tumoral.
* `ct_resampled.nii.gz`e `seg_resampled.nii.gz`: reamostragem espacial isotrópica dos volumes para resolução uniforme de $1,0 \times 1,0 \times 1,0\text{ mm}^3$ (utilizando interpolação linear para a TC e nearest neighbor para a máscara).
* `ct_multi.npy` e `seg.npy`: volumes finais padronizados em matriz cúbica de $128 \times 128 \times 128$ voxels após recorte centrado na região de interesse (ROI), janelamento HU (tecido mole e pulmonar) e normalização de intensidades para o intervalo $[0, 1]$

In [11]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [12]:
!pip -q install SimpleITK

In [13]:
import ast
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from tqdm import tqdm
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
BASE_DIR = "/content/gdrive/MyDrive/NSCLC/NSCLC-Radiomics/"
CLINICAL_CSV = "/content/gdrive/MyDrive/NSCLC/df_clinical.csv"

df_clin = pd.read_csv(CLINICAL_CSV)
pacientes = sorted(df_clin["PatientID"].astype(str).unique())

resultados = []

for patient_id in pacientes:

    patient_dir = os.path.join(BASE_DIR, patient_id)

    arquivos = {
        "ct_aligned": os.path.join(
            patient_dir, "nii_aligned", "ct_aligned.nii.gz"
        ),
        "seg_aligned": os.path.join(
            patient_dir, "nii_aligned", "seg_aligned.nii.gz"
        ),
        "ct_resampled": os.path.join(
            patient_dir, "resampled", "ct_resampled.nii.gz"
        ),
        "seg_resampled": os.path.join(
            patient_dir, "resampled", "seg_resampled.nii.gz"
        ),
        "ct_final": os.path.join(
            patient_dir, "processed_final", "ct_multi.npy"
        ),
        "seg_final": os.path.join(
            patient_dir, "processed_final", "seg.npy"
        )
    }

    row = {"PatientID": patient_id}

    for nome, caminho in arquivos.items():

        row[f"{nome}_exists"] = os.path.exists(caminho)

        if not os.path.exists(caminho):
            continue

        try:

            if caminho.endswith(".npy"):
                arr = np.load(caminho, allow_pickle=False)
                # Registra shape e dtype
                row[f"{nome}_shape"] = str(arr.shape)
                row[f"{nome}_dtype"] = str(arr.dtype)

                if np.issubdtype(arr.dtype, np.number):

                    row[f"{nome}_finite"] = bool(
                        np.isfinite(arr).all()
                    )

                    if np.isfinite(arr).any():

                        row[f"{nome}_min"] = float(
                            np.nanmin(arr)
                        )

                        row[f"{nome}_max"] = float(
                            np.nanmax(arr)
                        )

                if nome == "seg_final":
                    # 20 Valores únicos para a segmentação
                    row["seg_final_unique"] = str(
                        np.unique(arr)[:20]
                    )

                    row["seg_final_voxels"] = int(
                        np.count_nonzero(arr > 0)
                    )

            else:
                """
                Para os arquivos nii, registra os metadados geométricos,
                ou seja, size, spacing, origin, direction e dtype
                """
                img = sitk.ReadImage(caminho)

                row[f"{nome}_size"] = str(
                    img.GetSize()
                )

                row[f"{nome}_spacing"] = str(
                    img.GetSpacing()
                )

                row[f"{nome}_origin"] = str(
                    img.GetOrigin()
                )

                row[f"{nome}_direction"] = str(
                    img.GetDirection()
                )

                row[f"{nome}_dtype"] = str(
                    img.GetPixelIDTypeAsString()
                )

        except Exception as e:

            row[f"{nome}_error"] = str(e)

    resultados.append(row)

df_inspecao = pd.DataFrame(resultados)

display(df_inspecao.head(10))
df_inspecao.to_csv("/content/gdrive/MyDrive/NSCLC/df_inspecao.csv", index=False)

,PatientID,ct_aligned_exists,ct_aligned_size,ct_aligned_spacing,ct_aligned_origin,ct_aligned_direction,ct_aligned_dtype,seg_aligned_exists,seg_aligned_size,seg_aligned_spacing,...,ct_final_min,ct_final_max,seg_final_exists,seg_final_shape,seg_final_dtype,seg_final_finite,seg_final_min,seg_final_max,seg_final_unique,seg_final_voxels
0,LUNG1-001,True,"(512, 512, 134)","(0.9765625, 0.9765625, 3.0)","(-249.51171875, -460.51171875, -681.5)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",32-bit signed integer,True,"(512, 512, 134)","(0.9765625, 0.9765625, 3.0)",...,0.0,1.0,True,"(128, 128, 128)",uint8,True,0.0,1.0,[0 1],224394
1,LUNG1-002,True,"(512, 512, 111)","(0.9769999980926514, 0.9769999980926514, 3.0)","(-250.11199951171875, -250.11199951171875, -13...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",16-bit signed integer,True,"(512, 512, 111)","(0.9769999980926514, 0.9769999980926514, 3.0)",...,0.0,1.0,True,"(128, 128, 128)",uint8,True,0.0,1.0,[0 1],45010
2,LUNG1-003,True,"(512, 512, 107)","(0.9769999980926514, 0.9769999980926514, 3.0)","(-250.11199951171875, -250.11199951171875, -30...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",16-bit signed integer,True,"(512, 512, 107)","(0.9769999980926514, 0.9769999980926514, 3.0)",...,0.0,1.0,True,"(128, 128, 128)",uint8,True,0.0,1.0,[0 1],13257
3,LUNG1-004,True,"(512, 512, 114)","(0.9765625, 0.9765625, 3.0)","(-249.51170349121094, -249.51170349121094, -64...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",32-bit signed integer,True,"(512, 512, 114)","(0.9765625, 0.9765625, 3.0)",...,0.0,1.0,True,"(128, 128, 128)",uint8,True,0.0,1.0,[0 1],6786
4,LUNG1-005,True,"(512, 512, 91)","(0.9769999980926514, 0.9769999980926514, 3.0)","(-250.11199951171875, -250.11199951171875, -17...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",16-bit signed integer,True,"(512, 512, 91)","(0.9769999980926514, 0.9769999980926514, 3.0)",...,0.0,1.0,True,"(128, 128, 128)",uint8,True,0.0,1.0,[0 1],29871
5,LUNG1-006,True,"(512, 512, 114)","(0.9769999980926514, 0.9769999980926514, 3.0)","(-250.11199951171875, -250.11199951171875, -19...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",16-bit signed integer,True,"(512, 512, 114)","(0.9769999980926514, 0.9769999980926514, 3.0)",...,0.0,1.0,True,"(128, 128, 128)",uint8,True,0.0,1.0,[0 1],28215
6,LUNG1-007,True,"(512, 512, 129)","(0.9765625, 0.9765625, 3.0)","(-249.51171875, -436.51171875, -641.5)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",16-bit signed integer,True,"(512, 512, 129)","(0.9765625, 0.9765625, 3.0)",...,0.0,1.0,True,"(128, 128, 128)",uint8,True,0.0,0.0,[0],0
7,LUNG1-008,True,"(512, 512, 114)","(0.9769999980926514, 0.9769999980926514, 3.0)","(-250.11199951171875, -250.11199951171875, -19...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",16-bit signed integer,True,"(512, 512, 114)","(0.9769999980926514, 0.9769999980926514, 3.0)",...,0.0,1.0,True,"(128, 128, 128)",uint8,True,0.0,1.0,[0 1],9124
8,LUNG1-009,True,"(512, 512, 105)","(0.9769999980926514, 0.9769999980926514, 3.0)","(-250.11199951171875, -250.11199951171875, -16...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",16-bit signed integer,True,"(512, 512, 105)","(0.9769999980926514, 0.9769999980926514, 3.0)",...,0.0,1.0,True,"(128, 128, 128)",uint8,True,0.0,1.0,[0 1],2029
9,LUNG1-010,True,"(512, 512, 91)","(0.9769999980926514, 0.9769999980926514, 3.0)","(-250.11199951171875, -250.11199951171875, -11...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",16-bit signed integer,True,"(512, 512, 91)","(0.9769999980926514, 0.9769999980926514, 3.0)",...,0.0,1.0,True,"(128, 128, 128)",uint8,True,0.0,1.0,[0 1],14273


In [ ]:
print("SHAPES CT FINAL:")
display(
    df_inspecao["ct_final_shape"]
    .value_counts(dropna=False)
)

print("\nSHAPES SEG FINAL:")
display(
    df_inspecao["seg_final_shape"]
    .value_counts(dropna=False)
)

print("\nSHAPES CT ALIGNED:")
display(
    df_inspecao["ct_aligned_size"]
    .value_counts(dropna=False)
)

print("\nSHAPES CT RESAMPLED:")
display(
    df_inspecao["ct_resampled_size"]
    .value_counts(dropna=False)
)

SHAPES CT FINAL:


,count
ct_final_shape,
"(128, 128, 128, 2)",421



SHAPES SEG FINAL:


,count
seg_final_shape,
"(128, 128, 128)",421



SHAPES CT ALIGNED:


,count
ct_aligned_size,
"(512, 512, 134)",107
"(512, 512, 94)",34
"(512, 512, 131)",12
"(512, 512, 135)",11
"(512, 512, 176)",11
"(512, 512, 114)",11
"(512, 512, 132)",10
"(512, 512, 112)",9
"(512, 512, 130)",9



SHAPES CT RESAMPLED:


,count
ct_resampled_size,
"(500, 500, 402)",108
"(500, 500, 282)",34
"(500, 500, 393)",12
"(500, 500, 342)",11
"(500, 500, 528)",11
"(500, 500, 405)",10
"(500, 500, 396)",10
"(500, 500, 336)",9
"(500, 500, 390)",9


In [ ]:
output_path = os.path.join(
    BASE_DIR,
    "shapes.csv"
)

df_inspecao.to_csv(output_path, index=False)

print(f"Salvo em: {output_path}")

Salvo em: /content/gdrive/MyDrive/NSCLC/NSCLC-Radiomics/shapes.csv


In [15]:
df_shape = pd.read_csv("/content/gdrive/MyDrive/NSCLC/Dados_Imagens/shapes.csv")

In [16]:
df_shape.head()

,PatientID,ct_aligned_exists,ct_aligned_size,ct_aligned_spacing,ct_aligned_origin,ct_aligned_direction,ct_aligned_dtype,seg_aligned_exists,seg_aligned_size,seg_aligned_spacing,seg_aligned_origin,seg_aligned_direction,seg_aligned_dtype,ct_resampled_exists,ct_resampled_size,ct_resampled_spacing,ct_resampled_origin,ct_resampled_direction,ct_resampled_dtype,seg_resampled_exists,seg_resampled_size,seg_resampled_spacing,seg_resampled_origin,seg_resampled_direction,seg_resampled_dtype,ct_final_exists,ct_final_shape,ct_final_dtype,ct_final_finite,ct_final_min,ct_final_max,seg_final_exists,seg_final_shape,seg_final_dtype,seg_final_finite,seg_final_min,seg_final_max,seg_final_unique,seg_final_voxels
0,LUNG1-001,True,"(512, 512, 134)","(0.9765625, 0.9765625, 3.0)","(-249.51171875, -460.51171875, -681.5)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",32-bit signed integer,True,"(512, 512, 134)","(0.9765625, 0.9765625, 3.0)","(-249.51171875, -460.51171875, -681.5)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",8-bit unsigned integer,True,"(500, 500, 402)","(1.0, 1.0, 1.0)","(-249.51171875, -460.51171875, -681.5)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",32-bit signed integer,True,"(500, 500, 402)","(1.0, 1.0, 1.0)","(-249.51171875, -460.51171875, -681.5)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",8-bit unsigned integer,True,"(128, 128, 128, 2)",float32,True,0.0,1.0,True,"(128, 128, 128)",uint8,True,0.0,1.0,[0 1],224394
1,LUNG1-002,True,"(512, 512, 111)","(0.9769999980926514, 0.9769999980926514, 3.0)","(-250.11199951171875, -250.11199951171875, -13...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",16-bit signed integer,True,"(512, 512, 111)","(0.9769999980926514, 0.9769999980926514, 3.0)","(-250.11199951171875, -250.11199951171875, -13...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",8-bit unsigned integer,True,"(500, 500, 333)","(1.0, 1.0, 1.0)","(-250.11199951171875, -250.11199951171875, -13...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",16-bit signed integer,True,"(500, 500, 333)","(1.0, 1.0, 1.0)","(-250.11199951171875, -250.11199951171875, -13...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",8-bit unsigned integer,True,"(128, 128, 128, 2)",float32,True,0.0,1.0,True,"(128, 128, 128)",uint8,True,0.0,1.0,[0 1],45010
2,LUNG1-003,True,"(512, 512, 107)","(0.9769999980926514, 0.9769999980926514, 3.0)","(-250.11199951171875, -250.11199951171875, -30...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",16-bit signed integer,True,"(512, 512, 107)","(0.9769999980926514, 0.9769999980926514, 3.0)","(-250.11199951171875, -250.11199951171875, -30...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",8-bit unsigned integer,True,"(500, 500, 321)","(1.0, 1.0, 1.0)","(-250.11199951171875, -250.11199951171875, -30...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",16-bit signed integer,True,"(500, 500, 321)","(1.0, 1.0, 1.0)","(-250.11199951171875, -250.11199951171875, -30...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",8-bit unsigned integer,True,"(128, 128, 128, 2)",float32,True,0.0,1.0,True,"(128, 128, 128)",uint8,True,0.0,1.0,[0 1],13257
3,LUNG1-004,True,"(512, 512, 114)","(0.9765625, 0.9765625, 3.0)","(-249.51170349121094, -249.51170349121094, -64...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",32-bit signed integer,True,"(512, 512, 114)","(0.9765625, 0.9765625, 3.0)","(-249.51170349121094, -249.51170349121094, -64...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",8-bit unsigned integer,True,"(500, 500, 342)","(1.0, 1.0, 1.0)","(-249.51170349121094, -249.51170349121094, -64...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",32-bit signed integer,True,"(500, 500, 342)","(1.0, 1.0, 1.0)","(-249.51170349121094, -249.51170349121094, -64...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",8-bit unsigned integer,True,"(128, 128, 128, 2)",float32,True,0.0,1.0,True,"(128, 128, 128)",uint8,True,0.0,1.0,[0 1],6786
4,LUNG1-005,True,"(512, 512, 91)","(0.9769999980926514, 0.9769999980926514, 3.0)","(-250.11199951171875, -250

In [17]:
df_shape.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421 entries, 0 to 420
Data columns (total 39 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   PatientID                421 non-null    object 
 1   ct_aligned_exists        421 non-null    bool   
 2   ct_aligned_size          421 non-null    object 
 3   ct_aligned_spacing       421 non-null    object 
 4   ct_aligned_origin        421 non-null    object 
 5   ct_aligned_direction     421 non-null    object 
 6   ct_aligned_dtype         421 non-null    object 
 7   seg_aligned_exists       421 non-null    bool   
 8   seg_aligned_size         421 non-null    object 
 9   seg_aligned_spacing      421 non-null    object 
 10  seg_aligned_origin       421 non-null    object 
 11  seg_aligned_direction    421 non-null    object 
 12  seg_aligned_dtype        421 non-null    object 
 13  ct_resampled_exists      421 non-null    bool   
 14  ct_resampled_size        4

In [18]:
print(df_shape["ct_final_shape"].value_counts(dropna=False))
print(df_shape["seg_final_voxels"].eq(0).sum(), "ROIs vazias no df_inspecao")

ct_final_shape
(128, 128, 128, 2)    421
Name: count, dtype: int64
52 ROIs vazias no df_inspecao


In [ ]:
# Verificação complementar

registros_nii = []
registros_npy = []

for patient_id in tqdm(pacientes):

    patient_dir = os.path.join(BASE_DIR, str(patient_id))

    # NIFTI
    caminhos_nii = {
        "ct_aligned": os.path.join(
            patient_dir, "nii_aligned", "ct_aligned.nii.gz"
        ),
        "seg_aligned": os.path.join(
            patient_dir, "nii_aligned", "seg_aligned.nii.gz"
        ),
        "ct_resampled": os.path.join(
            patient_dir, "resampled", "ct_resampled.nii.gz"
        ),
        "seg_resampled": os.path.join(
            patient_dir, "resampled", "seg_resampled.nii.gz"
        )
    }

    imagens = {}

    for nome, caminho in caminhos_nii.items():

        if not os.path.exists(caminho):
            continue

        try:

            img = sitk.ReadImage(caminho)

            imagens[nome] = img

            registros_nii.append({
                "PatientID": patient_id,
                "arquivo": nome,
                "size": str(img.GetSize()),
                "spacing": str(img.GetSpacing()),
                "origin": str(img.GetOrigin()),
                "direction": str(img.GetDirection()),
                "dtype": img.GetPixelIDTypeAsString()
            })

        except Exception as e:

            registros_nii.append({
                "PatientID": patient_id,
                "arquivo": nome,
                "erro": str(e)
            })

    caminhos_npy = {
        "ct_multi": os.path.join(
            patient_dir, "processed_final", "ct_multi.npy"
        ),
        "seg": os.path.join(
            patient_dir, "processed_final", "seg.npy"
        )
    }

    for nome, caminho in caminhos_npy.items():

        if not os.path.exists(caminho):
            continue
        try:

            arr = np.load(
                caminho,
                allow_pickle=False
            )

            registro = {
                "PatientID": patient_id,
                "arquivo": nome,
                "shape": str(arr.shape),
                "dtype": str(arr.dtype),
                "finite": bool(np.isfinite(arr).all())
            }

            if np.isfinite(arr).any():

                registro["min"] = float(np.nanmin(arr))
                registro["max"] = float(np.nanmax(arr))
                registro["mean"] = float(np.nanmean(arr))
                registro["std"] = float(np.nanstd(arr))

                # Valores únicos para arrays com poucos valores
                if nome == "seg":

                    unique_vals = np.unique(arr)

                    registro["unique_values"] = str(
                        unique_vals[:20]
                    )

                    registro["n_unique"] = len(unique_vals)

                    registro["n_voxels_roi"] = int(
                        np.count_nonzero(arr > 0)
                    )

                # Para CT: valores únicos
                else:

                    registro["n_unique"] = int(
                        np.unique(arr).size
                    )

                    registro["fraction_zero"] = float(
                        np.mean(arr == 0)
                    )

                    registro["fraction_one"] = float(
                        np.mean(arr == 1)
                    )

            registros_npy.append(registro)

        except Exception as e:

            registros_npy.append({
                "PatientID": patient_id,
                "arquivo": nome,
                "erro": str(e)
            })


df_nii_meta = pd.DataFrame(registros_nii)
df_npy_meta = pd.DataFrame(registros_npy)



100%|██████████| 421/421 [1:14:19<00:00, 10.59s/it]


In [ ]:
print("Espaçamentos NIFTI:")

display(
    df_nii_meta.groupby(
        ["arquivo", "spacing"],
        dropna=False
    ).size().reset_index(name="count")
)

Espaçamentos NIFTI:


,arquivo,spacing,count
0,ct_aligned,"(0.7207000255584717, 0.7207000255584717, 3.0)",1
1,ct_aligned,"(0.8119999766349792, 0.8119999766349792, 3.0)",1
2,ct_aligned,"(0.8980000019073486, 0.8980000019073486, 3.0)",1
3,ct_aligned,"(0.912109375, 0.912109375, 3.0)",1
4,ct_aligned,"(0.9765620231628418, 0.9765620231628418, 3.0)",1
5,ct_aligned,"(0.9765625, 0.9765625, 2.9739129543304443)",1
6,ct_aligned,"(0.9765625, 0.9765625, 2.977612018585205)",1
7,ct_aligned,"(0.9765625, 0.9765625, 3.0)",320
8,ct_aligned,"(0.9765999913215637, 0.9765999913215637, 3.0)",1
9,ct_aligned,"(0.9769999980926514, 0.9769999980926514, 2.971...",1


In [ ]:
print("SHAPES SEG ALIGNED:")

display(
    df_nii_meta[
        df_nii_meta["arquivo"] == "seg_aligned"
    ]["size"].value_counts()
)

print("\nSHAPES SEG RESAMPLED:")

display(
    df_nii_meta[
        df_nii_meta["arquivo"] == "seg_resampled"
    ]["size"].value_counts()
)

SHAPES SEG ALIGNED:


,count
size,
"(512, 512, 134)",107
"(512, 512, 94)",34
"(512, 512, 131)",12
"(512, 512, 135)",11
"(512, 512, 176)",11
"(512, 512, 114)",11
"(512, 512, 132)",10
"(512, 512, 112)",9
"(512, 512, 130)",9



SHAPES SEG RESAMPLED:


,count
size,
"(500, 500, 402)",108
"(500, 500, 282)",34
"(500, 500, 393)",12
"(500, 500, 342)",11
"(500, 500, 528)",11
"(500, 500, 405)",10
"(500, 500, 396)",10
"(500, 500, 336)",9
"(500, 500, 390)",9


In [ ]:
print("CT MULTI:")

display(
    df_npy_meta[
        df_npy_meta["arquivo"] == "ct_multi"
    ][
        ["dtype", "min", "max", "mean", "std", "n_unique"]
    ].describe(include="all")
)

print("\nSEG FINAL:")

display(
    df_npy_meta[
        df_npy_meta["arquivo"] == "seg"
    ][
        ["dtype", "min", "max", "n_unique",
         "unique_values", "n_voxels_roi"]
    ].head(10)
)

CT MULTI:


,dtype,min,max,mean,std,n_unique
count,421,421.0,421.0,421.000000,421.000000,421.000000
unique,1,NaN,NaN,NaN,NaN,NaN
top,float32,NaN,NaN,NaN,NaN,NaN
freq,421,NaN,NaN,NaN,NaN,NaN
mean,NaN,0.0,1.0,0.388170,0.344208,1200.294537
std,NaN,0.0,0.0,0.074330,0.024567,10.227121
min,NaN,0.0,1.0,0.011492,0.092278,1007.000000
25%,NaN,0.0,1.0,0.348468,0.335921,1201.000000
50%,NaN,0.0,1.0,0.385549,0.341935,1201.000000
75%,NaN,0.0,1.0,0.428711,0.351928,1201.000000



SEG FINAL:


,dtype,min,max,n_unique,unique_values,n_voxels_roi
1,uint8,0.0,1.0,2,[0 1],224394.0
3,uint8,0.0,1.0,2,[0 1],45010.0
5,uint8,0.0,1.0,2,[0 1],13257.0
7,uint8,0.0,1.0,2,[0 1],6786.0
9,uint8,0.0,1.0,2,[0 1],29871.0
11,uint8,0.0,1.0,2,[0 1],28215.0
13,uint8,0.0,0.0,1,[0],0.0
15,uint8,0.0,1.0,2,[0 1],9124.0
17,uint8,0.0,1.0,2,[0 1],2029.0
19,uint8,0.0,1.0,2,[0 1],14273.0


In [ ]:
# Pacientes com ROIs vazias

pacientes_roi_vazia = df_npy_meta[
    (df_npy_meta["arquivo"] == "seg") &
    (df_npy_meta["n_voxels_roi"] == 0)
]["PatientID"].tolist()

print("Pacientes com ROI final vazia:")
print(pacientes_roi_vazia)

print("\nTotal de pacientes com ROI final vazia:")
print(len(pacientes_roi_vazia))

Pacientes com ROI final vazia:
['LUNG1-007', 'LUNG1-018', 'LUNG1-037', 'LUNG1-061', 'LUNG1-073', 'LUNG1-078', 'LUNG1-082', 'LUNG1-088', 'LUNG1-098', 'LUNG1-108', 'LUNG1-110', 'LUNG1-150', 'LUNG1-154', 'LUNG1-159', 'LUNG1-160', 'LUNG1-164', 'LUNG1-167', 'LUNG1-170', 'LUNG1-171', 'LUNG1-176', 'LUNG1-180', 'LUNG1-184', 'LUNG1-192', 'LUNG1-198', 'LUNG1-203', 'LUNG1-208', 'LUNG1-216', 'LUNG1-231', 'LUNG1-240', 'LUNG1-247', 'LUNG1-255', 'LUNG1-265', 'LUNG1-267', 'LUNG1-272', 'LUNG1-274', 'LUNG1-275', 'LUNG1-295', 'LUNG1-300', 'LUNG1-313', 'LUNG1-318', 'LUNG1-326', 'LUNG1-342', 'LUNG1-359', 'LUNG1-379', 'LUNG1-383', 'LUNG1-385', 'LUNG1-388', 'LUNG1-391', 'LUNG1-395', 'LUNG1-403', 'LUNG1-415', 'LUNG1-420']

Total de pacientes com ROI final vazia:
52


In [ ]:
# Comparação de voxels entre as etapas

def comparar_segmentacoes(patient_id):

    patient_dir = os.path.join(BASE_DIR, str(patient_id))

    caminhos = {
        "aligned": os.path.join(
            patient_dir,
            "nii_aligned",
            "seg_aligned.nii.gz"
        ),

        "resampled": os.path.join(
            patient_dir,
            "resampled",
            "seg_resampled.nii.gz"
        ),

        "final": os.path.join(
            patient_dir,
            "processed_final",
            "seg.npy"
        )
    }

    resultado = {
        "PatientID": patient_id
    }

    for etapa, caminho in caminhos.items():

        if not os.path.exists(caminho):

            resultado[f"{etapa}_exists"] = False
            resultado[f"{etapa}_voxels"] = np.nan

            continue

        resultado[f"{etapa}_exists"] = True

        if etapa == "final":

            arr = np.load(caminho)

        else:

            img = sitk.ReadImage(caminho)

            arr = sitk.GetArrayFromImage(img)

        resultado[f"{etapa}_voxels"] = int(
            np.count_nonzero(arr > 0)
        )

        if etapa != "final":

            resultado[f"{etapa}_spacing"] = str(
                img.GetSpacing()
            )

    return resultado


comparacao_roi = pd.DataFrame([
    comparar_segmentacoes(p)
    for p in pacientes_roi_vazia
])

display(comparacao_roi)

,PatientID,aligned_exists,aligned_voxels,aligned_spacing,resampled_exists,resampled_voxels,resampled_spacing,final_exists,final_voxels
0,LUNG1-007,True,91762,"(0.9765625, 0.9765625, 3.0)",True,262662,"(1.0, 1.0, 1.0)",True,0
1,LUNG1-018,True,244968,"(0.9769999980926514, 0.9769999980926514, 3.0)",True,701634,"(1.0, 1.0, 1.0)",True,0
2,LUNG1-037,True,215440,"(0.9769999980926514, 0.9769999980926514, 3.0)",True,616401,"(1.0, 1.0, 1.0)",True,0
3,LUNG1-061,True,246151,"(0.9769999980926514, 0.9769999980926514, 3.0)",True,705972,"(1.0, 1.0, 1.0)",True,0
4,LUNG1-073,True,214652,"(0.9765625, 0.9765625, 3.0)",True,614292,"(1.0, 1.0, 1.0)",True,0
5,LUNG1-078,True,80656,"(0.9769999980926514, 0.9769999980926514, 3.0)",True,230775,"(1.0, 1.0, 1.0)",True,0
6,LUNG1-082,True,212458,"(0.9765625, 0.9765625, 3.0)",True,608925,"(1.0, 1.0, 1.0)",True,0
7,LUNG1-088,True,410521,"(0.9769999980926514, 0.9769999980926514, 3.0)",True,1175562,"(1.0, 1.0, 1.0)",True,0
8,LUNG1-098,True,252384,"(0.9765625, 0.9765625, 3.0)",True,722721,"(1.0, 1.0, 1.0)",True,0
9,LUNG1-108,True,257167,"(0.9765625, 0.9765625, 3.0)",True,737130,"(1.0, 1.0, 1.0)",True,0


In [ ]:
# Definindo o diretório de saída que não estava configurado nesta célula
OUT_DIR = "/content/gdrive/MyDrive/NSCLC/Dados_Imagens/"
os.makedirs(OUT_DIR, exist_ok=True)

df_nii_meta.to_csv(
    os.path.join(
        OUT_DIR,
        "metadados_nii.csv"
    ),
    index=False
)

df_npy_meta.to_csv(
    os.path.join(
        OUT_DIR,
        "estatisticas_npy.csv"
    ),
    index=False
)

In [ ]:
BASE_DIR = "/content/gdrive/MyDrive/NSCLC/NSCLC-Radiomics/"
CLINICAL_CSV = "/content/gdrive/MyDrive/NSCLC/df_clinical.csv"
AUDIT_OUT_DIR = "/content/gdrive/MyDrive/NSCLC/Dados_Imagens/"

os.makedirs(AUDIT_OUT_DIR, exist_ok=True)
PNG_SAMPLES_DIR = os.path.join(AUDIT_OUT_DIR, "amostras_visuais")
os.makedirs(PNG_SAMPLES_DIR, exist_ok=True)

def auditar_paciente(patient_id):
    patient_dir = os.path.join(BASE_DIR, patient_id)
    nii_dir = os.path.join(patient_dir, "nii_aligned")
    resampled_dir = os.path.join(patient_dir, "resampled")
    final_dir = os.path.join(patient_dir, "processed_final")

    res = {
        "PatientID": patient_id,
        "has_ct_aligned": False,
        "has_seg_aligned": False,
        "aligned_geom_ok": False,
        "has_resampled": False,
        "resampled_spacing_ok": False,
        "has_final_npy": False,
        "seg_has_voxels": False,
        "roi_empty": True,
        "lesion_inside_crop": False,
        "final_shape_ok": False,
        "soft_intensity_ok": False,
        "lung_intensity_ok": False,
        "tumor_volume_voxels": 0,
        "atypical_flag": False,
        "error_msg": ""
    }

    # Verificar TC e SEG alinhadas
    ct_nii = os.path.join(nii_dir, "ct_aligned.nii.gz")
    seg_nii = os.path.join(nii_dir, "seg_aligned.nii.gz")

    if os.path.exists(ct_nii): res["has_ct_aligned"] = True
    if os.path.exists(seg_nii): res["has_seg_aligned"] = True

    if res["has_ct_aligned"] and res["has_seg_aligned"]:
        try:
            img_ct = sitk.ReadImage(ct_nii)
            img_seg = sitk.ReadImage(seg_nii)

            # Checar alinhamento geométrico (Origem, Tamanho, Espaçamento, Direção)
            same_size = img_ct.GetSize() == img_seg.GetSize()
            same_origin = np.allclose(img_ct.GetOrigin(), img_seg.GetOrigin(), atol=1e-3)
            same_spacing = np.allclose(img_ct.GetSpacing(), img_seg.GetSpacing(), atol=1e-3)
            same_direction = np.allclose(img_ct.GetDirection(), img_seg.GetDirection(), atol=1e-3)

            res["aligned_geom_ok"] = (same_size and same_origin and same_spacing and same_direction)
        except Exception as e:
            res["error_msg"] += f"Erro leitura NIfTI: {str(e)}; "

    # Verificar reamostragem (1x1x1 mm)
    ct_res = os.path.join(resampled_dir, "ct_resampled.nii.gz")
    seg_res = os.path.join(resampled_dir, "seg_resampled.nii.gz")

    if os.path.exists(ct_res) and os.path.exists(seg_res):
        res["has_resampled"] = True
        try:
            r_ct = sitk.ReadImage(ct_res)
            sp = r_ct.GetSpacing()
            res["resampled_spacing_ok"] = np.allclose(sp, (1.0, 1.0, 1.0), atol=1e-2)
        except Exception as e:
            res["error_msg"] += f"Erro reamostragem: {str(e)}; "

    # Verificar volumes processados finais (.npy)
    ct_final_path = os.path.join(final_dir, "ct_multi.npy")
    seg_final_path = os.path.join(final_dir, "seg.npy")

    if os.path.exists(ct_final_path) and os.path.exists(seg_final_path):
        res["has_final_npy"] = True
        try:
            ct_arr = np.load(ct_final_path)
            seg_arr = np.load(seg_final_path)

            # Checar Voxels da Máscara
            num_voxels = int(np.sum(seg_arr > 0))
            res["tumor_volume_voxels"] = num_voxels
            res["seg_has_voxels"] = num_voxels > 0
            res["roi_empty"] = num_voxels == 0

            # Checar Shape Fixo (128, 128, 128)
            spatial_shape = seg_arr.shape
            res["final_shape_ok"] = (spatial_shape == (128, 128, 128))

            # Checar se a lesão está totalmente contida no crop (não toca nas bordas extremas)
            if res["seg_has_voxels"]:
                coords = np.argwhere(seg_arr > 0)
                touches_border = (
                    np.any(coords[:, 0] == 0) or np.any(coords[:, 0] == spatial_shape[0] - 1) or
                    np.any(coords[:, 1] == 0) or np.any(coords[:, 1] == spatial_shape[1] - 1) or
                    np.any(coords[:, 2] == 0) or np.any(coords[:, 2] == spatial_shape[2] - 1)
                )
                res["lesion_inside_crop"] = not touches_border

            # Checar intensidade dos canais [0, 1]
            soft_ch = ct_arr[..., 0] if ct_arr.ndim == 4 else ct_arr[0]
            lung_ch = ct_arr[..., 1] if ct_arr.ndim == 4 else ct_arr[1]

            res["soft_intensity_ok"] = bool((soft_ch.min() >= 0.0) and (soft_ch.max() <= 1.0))
            res["lung_intensity_ok"] = bool((lung_ch.min() >= 0.0) and (lung_ch.max() <= 1.0))

            # Identificar volumes atípicos
            if num_voxels < 100 or num_voxels > 150000 or touches_border or res["roi_empty"]:
                res["atypical_flag"] = True

        except Exception as e:
            res["error_msg"] += f"Erro arquivos .npy final: {str(e)}; "

    return res

def salvar_amostra_visual(patient_id, is_atypical=False):
    final_dir = os.path.join(BASE_DIR, patient_id, "processed_final")
    ct_p = os.path.join(final_dir, "ct_multi.npy")
    seg_p = os.path.join(final_dir, "seg.npy")

    if not (os.path.exists(ct_p) and os.path.exists(seg_p)):
        return

    ct = np.load(ct_p)
    seg = np.load(seg_p)

    if ct.ndim == 4 and ct.shape[-1] == 2:
        soft = ct[..., 0]
        lung = ct[..., 1]
    elif ct.ndim == 4 and ct.shape[0] == 2:
        soft = ct[0]
        lung = ct[1]
    else:
        soft, lung = ct[..., 0], ct[..., 1]

    if np.sum(seg) > 0:
        z_sums = np.sum(seg, axis=(1, 2)) if seg.ndim == 3 else np.sum(seg, axis=(0, 1))
        z_slice = int(np.argmax(z_sums))
    else:
        z_slice = seg.shape[0] // 2

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    prefix = "[ATÍPICO] " if is_atypical else ""
    fig.suptitle(f"{prefix}Auditoria Visual Paciente: {patient_id} (Slice Z={z_slice})", fontsize=12)

    axes[0].imshow(soft[z_slice] if soft.ndim==3 else soft[:,:,z_slice], cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("Canal Tecido Mole")
    axes[0].axis("off")

    axes[1].imshow(lung[z_slice] if lung.ndim==3 else lung[:,:,z_slice], cmap="bone", vmin=0, vmax=1)
    axes[1].set_title("Canal Pulmonar")
    axes[1].axis("off")

    axes[2].imshow(soft[z_slice] if soft.ndim==3 else soft[:,:,z_slice], cmap="gray", vmin=0, vmax=1)
    axes[2].imshow(seg[z_slice] if seg.ndim==3 else seg[:,:,z_slice], cmap="jet", alpha=0.5)
    axes[2].set_title("Overlay Tumor")
    axes[2].axis("off")

    plt.tight_layout()
    out_name = f"{'ATYPICAL_' if is_atypical else ''}{patient_id}_audit.png"
    plt.savefig(os.path.join(PNG_SAMPLES_DIR, out_name), dpi=150)
    plt.close()


if __name__ == "__main__":
    df_clin = pd.read_csv(CLINICAL_CSV)
    pacientes = sorted(df_clin["PatientID"].unique().tolist())

    print(f"Iniciando chegagem das imagens para {len(pacientes)} pacientes")
    relatorio = []

    for p in tqdm(pacientes):
        info = auditar_paciente(p)
        relatorio.append(info)

    df_audit = pd.DataFrame(relatorio)

    csv_path = os.path.join(AUDIT_OUT_DIR, "audit_report.csv")
    df_audit.to_csv(csv_path, index=False)

    resumo_executivo = {
        "total_pacientes_auditados": len(df_audit),
        "pacientes_com_ct_e_seg": int((df_audit["has_ct_aligned"] & df_audit["has_seg_aligned"]).sum()),
        "alinhamento_geometrico_ok": int(df_audit["aligned_geom_ok"].sum()),
        "reamostragem_1mm_ok": int(df_audit["resampled_spacing_ok"].sum()),
        "com_volumes_finais_npy": int(df_audit["has_final_npy"].sum()),
        "com_roi_vazia_erro": int(df_audit["roi_empty"].sum()),
        "lesao_totalmente_contida_no_crop": int(df_audit["lesion_inside_crop"].sum()),
        "shape_fixo_128_ok": int(df_audit["final_shape_ok"].sum()),
        "intensidade_normalizada_ok": int((df_audit["soft_intensity_ok"] & df_audit["lung_intensity_ok"]).sum()),
        "total_casos_atipicos": int(df_audit["atypical_flag"].sum())
    }

    json_path = os.path.join(AUDIT_OUT_DIR, "audit_summary.json")
    with open(json_path, "w") as f:
        json.dump(resumo_executivo, f, indent=4)

    atipicos = df_audit[df_audit["atypical_flag"]]["PatientID"].tolist()
    normais = df_audit[~df_audit["atypical_flag"]]["PatientID"].head(10).tolist()

    for p in atipicos:
        salvar_amostra_visual(p, is_atypical=True)
    for p in normais:
        salvar_amostra_visual(p, is_atypical=False)

    print("\n" + "="*60)
    print("RELATÓRIO")
    print(f"CSV Detalhado: {csv_path}")
    print(f"JSON Resumo: {json_path}")
    print(f"Amostras PNG: {PNG_SAMPLES_DIR}")
    print("="*60)
    print(json.dumps(resumo_executivo, indent=2))

Iniciando chegagem das imagens para 421 pacientes


100%|██████████| 421/421 [1:22:53<00:00, 11.81s/it]



RELATÓRIO
CSV Detalhado: /content/gdrive/MyDrive/NSCLC/Dados_Imagens/audit_report.csv
JSON Resumo: /content/gdrive/MyDrive/NSCLC/Dados_Imagens/audit_summary.json
Amostras PNG: /content/gdrive/MyDrive/NSCLC/Dados_Imagens/amostras_visuais
{
  "total_pacientes_auditados": 421,
  "pacientes_com_ct_e_seg": 421,
  "alinhamento_geometrico_ok": 421,
  "reamostragem_1mm_ok": 421,
  "com_volumes_finais_npy": 421,
  "com_roi_vazia_erro": 52,
  "lesao_totalmente_contida_no_crop": 36,
  "shape_fixo_128_ok": 421,
  "intensidade_normalizada_ok": 421,
  "total_casos_atipicos": 387
}
